# SCT 1.7B Convergence Experiment
**Spectral Compact Training** — SmolLM2-1.7B fine-tuning on Alpaca

Dense vs SCT head-to-head comparison with loss curves.

MLP compression at rank 32: **51x per layer**

---
**Runtime:** Select GPU via Runtime > Change runtime type > A100 (or T4/V100)

**Expected time:** ~1-2 hours total on A100, ~3-4 hours on T4

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers datasets matplotlib

In [ ]:
# Cell 2: Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 3: SpectralLinear implementation
import math
import torch
import torch.nn as nn


class SpectralLinear(nn.Module):
    """W = U diag(s) V^T with Stiefel QR retraction."""

    def __init__(self, in_features, out_features, rank=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = min(rank, min(in_features, out_features))

        U = torch.randn(in_features, self.rank) / math.sqrt(in_features)
        V = torch.randn(out_features, self.rank) / math.sqrt(out_features)
        Q_U, R_U = torch.linalg.qr(U)
        Q_V, R_V = torch.linalg.qr(V)
        self.U = nn.Parameter((Q_U * torch.sign(torch.diag(R_U)))[:, :self.rank])
        self.V = nn.Parameter((Q_V * torch.sign(torch.diag(R_V)))[:, :self.rank])
        self.s = nn.Parameter(torch.ones(self.rank))

    def forward(self, x):
        return (x @ self.U) * self.s @ self.V.T

    @classmethod
    def from_linear(cls, linear, rank=None, energy=None):
        """Convert nn.Linear to SpectralLinear via truncated SVD."""
        W = linear.weight.data.float()
        U_full, S_full, Vt_full = torch.linalg.svd(W, full_matrices=False)

        if energy is not None:
            total_energy = (S_full ** 2).sum()
            cumulative = torch.cumsum(S_full ** 2, dim=0) / total_energy
            k = max(1, int((cumulative >= energy).nonzero(as_tuple=True)[0][0].item()) + 1)
            if rank is not None:
                k = min(k, rank)
        elif rank is not None:
            k = min(rank, S_full.shape[0])
        else:
            k = min(32, S_full.shape[0])

        layer = cls.__new__(cls)
        nn.Module.__init__(layer)
        layer.in_features = linear.in_features
        layer.out_features = linear.out_features
        layer.rank = k
        layer.U = nn.Parameter(Vt_full[:k, :].T.contiguous())
        layer.V = nn.Parameter(U_full[:, :k].contiguous())
        layer.s = nn.Parameter(S_full[:k].contiguous())
        return layer

    @torch.no_grad()
    def retract(self):
        """QR retraction to Stiefel manifold."""
        for M in [self.U, self.V]:
            Q, R = torch.linalg.qr(M.data)
            M.data = (Q * torch.sign(torch.diag(R)))[:, :self.rank]


def retract_all(module):
    for m in module.modules():
        if isinstance(m, SpectralLinear):
            m.retract()


def convert_model_mlp_to_spectral(model, rank=None, energy=None):
    """Convert all MLP linear layers to SpectralLinear."""
    converted = 0
    for name, module in model.named_modules():
        for attr in ['gate_proj', 'up_proj', 'down_proj']:
            if hasattr(module, attr):
                linear = getattr(module, attr)
                if isinstance(linear, nn.Linear):
                    spectral = SpectralLinear.from_linear(linear, rank=rank, energy=energy)
                    setattr(module, attr, spectral)
                    converted += 1
    return converted


print("SpectralLinear ready.")

In [ ]:
# Cell 4: Load data
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B")
tokenizer.pad_token = tokenizer.eos_token

print("Loading Alpaca dataset...")
ds = load_dataset("tatsu-lab/alpaca", split="train")

MAX_LENGTH = 512

def format_sample(ex):
    if ex.get("input", "").strip():
        return f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

texts = [format_sample(ex) for ex in ds]
encodings = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                      padding="max_length", return_tensors="pt")

print(f"Samples: {encodings['input_ids'].shape[0]}  Seq length: {MAX_LENGTH}")

In [ ]:
# Cell 5: Training function
import time
import json

def train(model, encodings, device, steps, lr, batch_size, log_every,
          is_sct=False, label=""):
    """Train and return loss history."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    n_samples = input_ids.shape[0]

    losses = []
    step_times = []

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"  Training: {label}")
    print(f"  Steps: {steps}  LR: {lr}  Batch: {batch_size}")
    print(f"  Trainable params: {trainable:,}")
    print(f"{'='*60}\n")

    for step in range(1, steps + 1):
        idx = torch.randint(0, n_samples, (batch_size,))
        batch_input = input_ids[idx].to(device)
        batch_mask = attention_mask[idx].to(device)
        labels = batch_input.clone()
        labels[batch_mask == 0] = -100

        t0 = time.time()
        optimizer.zero_grad()
        outputs = model(input_ids=batch_input, attention_mask=batch_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if is_sct:
            retract_all(model)

        step_time = time.time() - t0
        loss_val = loss.item()
        losses.append({"step": step, "loss": loss_val, "time": step_time})
        step_times.append(step_time)

        if step % log_every == 0 or step == 1:
            ppl = math.exp(min(loss_val, 20))
            avg_time = sum(step_times[-log_every:]) / len(step_times[-log_every:])
            eta_min = (steps - step) * avg_time / 60
            if torch.cuda.is_available():
                mem = torch.cuda.max_memory_allocated() / 1e9
                print(f"  [{label}] Step {step:>5d}/{steps}  "
                      f"Loss: {loss_val:.4f}  PPL: {ppl:.2f}  "
                      f"Step: {avg_time:.2f}s  GPU: {mem:.1f}GB  ETA: {eta_min:.1f}min")
            else:
                print(f"  [{label}] Step {step:>5d}/{steps}  "
                      f"Loss: {loss_val:.4f}  PPL: {ppl:.2f}  "
                      f"Step: {avg_time:.2f}s  ETA: {eta_min:.1f}min")

    return losses

print("Training function ready.")

In [ ]:
# Cell 6: Configuration
# Adjust these as needed

STEPS = 2000          # Training steps per method
LR = 2e-5             # Learning rate
BATCH_SIZE = 4        # Batch size (A100 can handle 4, T4 use 2)
LOG_EVERY = 50        # Print every N steps
RANK = 32             # SCT rank
ENERGY = 0.95         # Energy retention for SVD truncation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config: {STEPS} steps, batch {BATCH_SIZE}, LR {LR}, rank {RANK}, energy {ENERGY}")
print(f"Device: {device}")

In [ ]:
# Cell 7: Dense baseline training
from transformers import AutoModelForCausalLM

print("Loading SmolLM2-1.7B for dense baseline...")
model_dense = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B",
    torch_dtype=torch.float32
).to(device)

dense_params = sum(p.numel() for p in model_dense.parameters())
print(f"Dense parameters: {dense_params:,}")

dense_losses = train(
    model_dense, encodings, device,
    steps=STEPS, lr=LR, batch_size=BATCH_SIZE,
    log_every=LOG_EVERY, is_sct=False, label="Dense"
)

# Save and free memory
with open("dense_losses.json", "w") as f:
    json.dump(dense_losses, f)
print("Dense results saved.")

del model_dense
torch.cuda.empty_cache()
print("Memory freed.")

In [ ]:
# Cell 8: SCT training
from transformers import AutoModelForCausalLM

print("Loading SmolLM2-1.7B for SCT conversion...")
model_sct = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B",
    torch_dtype=torch.float32
).to(device)

print(f"Converting MLP layers to SpectralLinear (energy={ENERGY})...")
n_converted = convert_model_mlp_to_spectral(model_sct, energy=ENERGY)
print(f"Converted {n_converted} layers")

sct_params = sum(p.numel() for p in model_sct.parameters() if p.requires_grad)
print(f"SCT parameters: {sct_params:,}")

# Report ranks
ranks = [m.rank for m in model_sct.modules() if isinstance(m, SpectralLinear)]
if ranks:
    print(f"Rank range: {min(ranks)}-{max(ranks)} (mean: {sum(ranks)/len(ranks):.0f})")

sct_losses = train(
    model_sct, encodings, device,
    steps=STEPS, lr=LR, batch_size=BATCH_SIZE,
    log_every=LOG_EVERY, is_sct=True, label="SCT"
)

with open("sct_losses.json", "w") as f:
    json.dump(sct_losses, f)
print("SCT results saved.")

del model_sct
torch.cuda.empty_cache()

In [ ]:
# Cell 9: Generate convergence plot
import matplotlib.pyplot as plt
import json
import math

with open("dense_losses.json") as f:
    dense_losses = json.load(f)
with open("sct_losses.json") as f:
    sct_losses = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

d_steps = [l["step"] for l in dense_losses]
d_loss = [l["loss"] for l in dense_losses]
s_steps = [l["step"] for l in sct_losses]
s_loss = [l["loss"] for l in sct_losses]

# Smoothed (rolling average of 20)
def smooth(vals, w=20):
    out = []
    for i in range(len(vals)):
        start = max(0, i - w + 1)
        out.append(sum(vals[start:i+1]) / (i - start + 1))
    return out

d_smooth = smooth(d_loss)
s_smooth = smooth(s_loss)

# Loss
ax1.plot(d_steps, d_loss, alpha=0.15, color='#2196F3')
ax1.plot(s_steps, s_loss, alpha=0.15, color='#FF5722')
ax1.plot(d_steps, d_smooth, label='Dense + AdamW', color='#2196F3', linewidth=2)
ax1.plot(s_steps, s_smooth, label=f'SCT (rank {RANK})', color='#FF5722', linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('SmolLM2-1.7B Fine-tuning: Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# PPL
d_ppl = [math.exp(min(l, 20)) for l in d_smooth]
s_ppl = [math.exp(min(l, 20)) for l in s_smooth]
ax2.plot(d_steps, d_ppl, label='Dense + AdamW', color='#2196F3', linewidth=2)
ax2.plot(s_steps, s_ppl, label=f'SCT (rank {RANK})', color='#FF5722', linewidth=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('Perplexity')
ax2.set_title('SmolLM2-1.7B Fine-tuning: Perplexity')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Footer
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
fig.text(0.5, 0.01,
         f'SmolLM2-1.7B | Alpaca | {gpu_name} | '
         f'Energy: {ENERGY} | MLP compression: 51x at rank 32 | EctoSpace/SCT',
         ha='center', fontsize=9, color='gray')

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('convergence_smollm2_1.7B_colab.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: convergence_smollm2_1.7B_colab.png")

In [ ]:
# Cell 10: Print summary
import math

d_final = dense_losses[-1]["loss"]
s_final = sct_losses[-1]["loss"]
d_ppl_final = math.exp(min(d_final, 20))
s_ppl_final = math.exp(min(s_final, 20))
ratio = s_ppl_final / d_ppl_final

d_time = sum(l["time"] for l in dense_losses)
s_time = sum(l["time"] for l in sct_losses)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
gpu_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"")
print(f"{'='*60}")
print(f"  CONVERGENCE RESULTS: SmolLM2-1.7B on Alpaca")
print(f"  Hardware: {gpu_name}")
print(f"{'='*60}")
print(f"  Dense final loss:  {d_final:.4f}  PPL: {d_ppl_final:.2f}")
print(f"  SCT final loss:    {s_final:.4f}  PPL: {s_ppl_final:.2f}")
print(f"  PPL ratio:         {ratio:.2f}x")
print(f"  Dense total time:  {d_time/60:.1f} min")
print(f"  SCT total time:    {s_time/60:.1f} min")
print(f"  Peak GPU memory:   {gpu_mem:.1f} GB")
print(f"  MLP compression:   51x at rank 32")
print(f"{'='*60}")

summary = {
    "model": "SmolLM2-1.7B",
    "dataset": "alpaca",
    "hardware": gpu_name,
    "steps": STEPS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "rank": RANK,
    "energy": ENERGY,
    "dense_final_loss": d_final,
    "dense_final_ppl": d_ppl_final,
    "sct_final_loss": s_final,
    "sct_final_ppl": s_ppl_final,
    "ppl_ratio": ratio,
    "dense_time_min": d_time / 60,
    "sct_time_min": s_time / 60,
    "peak_gpu_gb": gpu_mem
}

with open("summary_colab.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved: summary_colab.json")

In [ ]:
# Cell 11: Download results
from google.colab import files

files.download('convergence_smollm2_1.7B_colab.png')
files.download('dense_losses.json')
files.download('sct_losses.json')
files.download('summary_colab.json')
print("All files downloaded.")